# Mars TYXN Junction Classifier — Production Pipeline

## Quick Start

**What this does:** Trains and evaluates a stacking ensemble (RF + XGB + Gaussian-masked CNN) for classifying fracture network junctions as T (abutting), Y (triple), X (crossing), or N (non-junction) on Mars HiRISE skeleton images.

**Prerequisites:**
- Upload `marscracks_v7_bundle.zip` to Google Drive root
- GPU runtime enabled (Runtime → Change runtime type → GPU)

**Expected output:** Stacking meta-classifier with T F1 ≈ 0.57–0.62, macro F1 ≈ 0.77–0.78, accuracy ≈ 85%

**Key innovation:** Gaussian center mask (σ=40px) on 192×192 CNN patches forces attention on the target junction, eliminating distraction from neighboring junctions in dense fracture networks.

**Pipeline:**
1. Pre-train CNN on 24k old skeleton patches (transfer learning)
2. Fine-tune CNN on 4k matched-domain patches with Gaussian mask
3. Train RF and XGB on 31-dim geometry features
4. Stack all three via XGB meta-classifier
5. Evaluate on martian (667 held-out junctions)

---
## 1. Configuration

All key hyperparameters in one place. Modify here to experiment.

In [ ]:
# === CONFIGURATION ===

# Paths
DRIVE_ROOT = '/content/drive/MyDrive'
BUNDLE_NAME = 'marscracks_v7_bundle.zip'
OUTPUT_DIR_NAME = 'marscracks_v7_final'

# CNN architecture
CNN_ARCH = 'deeper_v2'          # DeeperCNN_GAP_v2: 6 conv, RF=86px, 833K params
CNN_PATCH_SIZE = 192            # 192×192 patches (native labeler output)
GAUSSIAN_SIGMA = 40             # Gaussian center mask sigma (pixels). 40 = sweet spot.

# CNN pre-training (on 24k old skeleton patches)
PRETRAIN_EPOCHS = 50
PRETRAIN_BATCH = 32
PRETRAIN_LR = 5e-4
PRETRAIN_PATIENCE = 10

# CNN fine-tuning (on 4k matched-domain patches)
FINETUNE_EPOCHS = 80
FINETUNE_BATCH = 32
FINETUNE_LR = 1e-4
FINETUNE_PATIENCE = 12

# Classical models (31-dim geometry features)
RF_ESTIMATORS = 500
RF_MAX_DEPTH = 12
XGB_ESTIMATORS = 500
XGB_MAX_DEPTH = 6
XGB_LR = 0.05
GEOMETRY_TRACE_LEN = 40         # Branch trace length for geometry features

# Stacking meta-classifier
META_ESTIMATORS = 100
META_MAX_DEPTH = 3
META_LR = 0.1

---
## 2. Setup

Mount Drive, install dependencies, unpack the training bundle.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost scikit-learn==1.5.2 scipy==1.13.1

import os, sys, shutil, zipfile, subprocess, pickle
from pathlib import Path

DRIVE = Path(DRIVE_ROOT)
BUNDLE_ZIP = DRIVE / BUNDLE_NAME
assert BUNDLE_ZIP.exists(), f'Bundle not found: {BUNDLE_ZIP}'

with zipfile.ZipFile(str(BUNDLE_ZIP), 'r') as z:
    z.extractall('/content')

SRC = Path('/content/marscracks_v7_bundle/src')
NEW_DATA = Path('/content/marscracks_v7_bundle/new_data')
OLD_DATA = Path('/content/marscracks_v7_bundle/old_data')
EVAL_DATA = Path('/content/marscracks_v7_bundle/eval_data')
MODELS = Path('/content/models')
MODELS.mkdir(exist_ok=True)

print(f'Source: {len(list(SRC.glob("*.py")))} files')
print(f'Train: {len(list((NEW_DATA / "image_patches").glob("*.png")))} patches')
print(f'Pretrain: {len(list((OLD_DATA / "image_patches").glob("*.png")))} patches')
print(f'Eval: {len(list((EVAL_DATA / "image_patches").glob("*.png")))} patches')

---
## 3. CNN Training (Gaussian Center Mask)

The CNN uses a Gaussian center mask (σ=40px) applied to 192×192 skeleton patches during both training and inference. This forces the network to focus on the target junction at patch center, preventing attention from spreading to neighboring junctions in dense fracture networks.

**Architecture:** DeeperCNN_GAP_v2 — 6 conv blocks, RF=86px, 128-dim GAP, 833K params.

**Training strategy:** Pre-train on 24k old (domain-shifted) patches for feature learning, then fine-tune on 4k matched-domain patches. The Gaussian mask is applied at both stages.

In [ ]:
# Pre-train CNN on 24k old patches with Gaussian center mask
cmd = (
    f"cd {SRC} && python train_cnn.py"
    f" --manifest {OLD_DATA}/manifest.csv --data-root {OLD_DATA}"
    f" --output-model {MODELS}/CNN_pretrained.pt"
    f" --arch {CNN_ARCH} --patch-size {CNN_PATCH_SIZE}"
    f" --epochs {PRETRAIN_EPOCHS} --batch-size {PRETRAIN_BATCH}"
    f" --lr {PRETRAIN_LR} --patience {PRETRAIN_PATIENCE}"
    f" --optimizer adam --scheduler plateau"
    f" --class-weight-mode balanced --label-smoothing 0.03"
    f" --train-rotate --train-flip --train-shift-max 8 --train-shift-prob 0.5"
    f" --dropout 0.3 --center-gaussian-sigma {GAUSSIAN_SIGMA}"
)
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-600:])
if result.returncode: print(result.stderr[-500:])

In [ ]:
# Fine-tune CNN on 4k matched-domain patches
cmd = (
    f"cd {SRC} && python train_cnn.py"
    f" --manifest {NEW_DATA}/manifest.csv --data-root {NEW_DATA}"
    f" --output-model {MODELS}/CNN_finetuned.pt"
    f" --arch {CNN_ARCH} --patch-size {CNN_PATCH_SIZE}"
    f" --epochs {FINETUNE_EPOCHS} --batch-size {FINETUNE_BATCH}"
    f" --lr {FINETUNE_LR} --patience {FINETUNE_PATIENCE}"
    f" --optimizer adam --scheduler plateau"
    f" --class-weight-mode balanced --label-smoothing 0.03"
    f" --train-rotate --train-flip --train-shift-max 8 --train-shift-prob 0.5"
    f" --dropout 0.3 --center-gaussian-sigma {GAUSSIAN_SIGMA}"
    f" --init-model {MODELS}/CNN_pretrained.pt --init-reset-head"
)
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-600:])
if result.returncode: print(result.stderr[-500:])

---
## 4. Classical Model Training

RF and XGB trained on 31-dim geometry features extracted from 192×192 patches. Features include collinearity at 7 radii (r12–r80), branch angles/gaps/lengths, T/Y template similarity, and derived collinearity consistency measures.

Both models use `--feature-regime geom_only` (no HOG — HOG features were shown to hurt performance on binary skeleton images).

In [ ]:
# Random Forest
cmd = (
    f"cd {SRC} && python train_rf.py"
    f" --manifest {NEW_DATA}/manifest.csv --data-root {NEW_DATA}"
    f" --output-model {MODELS}/RF.joblib"
    f" --patch-size {CNN_PATCH_SIZE} --feature-regime geom_only"
    f" --class-weight-mode balanced"
    f" --n-estimators {RF_ESTIMATORS} --max-depth {RF_MAX_DEPTH}"
    f" --geometry-trace-len {GEOMETRY_TRACE_LEN}"
)
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-400:])
if result.returncode: print(result.stderr[-500:])

# XGBoost
cmd = (
    f"cd {SRC} && python train_xgb.py"
    f" --manifest {NEW_DATA}/manifest.csv --data-root {NEW_DATA}"
    f" --output-model {MODELS}/XGB.joblib"
    f" --patch-size {CNN_PATCH_SIZE} --feature-regime geom_only"
    f" --class-weight-mode balanced"
    f" --max-depth {XGB_MAX_DEPTH} --learning-rate {XGB_LR}"
    f" --n-estimators {XGB_ESTIMATORS} --early-stopping-rounds 50"
    f" --geometry-trace-len {GEOMETRY_TRACE_LEN}"
)
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout[-400:])
if result.returncode: print(result.stderr[-500:])

---
## 5. Load Evaluation Data

martian evaluation set: 4 HiRISE skeleton images, 667 junctions (Y=451, T=66, X=35, N=115). Junctions were labeled on mit_b3 U-Net skeletons — same pipeline as training data.

In [ ]:
import csv, torch, joblib, numpy as np, cv2
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

sys.path.insert(0, str(SRC))
from classical_feature_builder import extract_geometry_feature_vector, GEOMETRY_FEATURE_NAMES
from train_cnn import DeeperCNN_GAP_v2, load_init_state

LABEL_MAP = {'N': 0, 'T': 1, 'X': 2, 'Y': 3}
IDX_TO_LABEL = ['N', 'T', 'X', 'Y']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load martian eval
eval_rows = list(csv.DictReader(open(EVAL_DATA / 'manifest.csv')))
patches_192, geom_features, true_labels = [], [], []
for r in eval_rows:
    p = cv2.imread(str(EVAL_DATA / r['relpath']), cv2.IMREAD_GRAYSCALE)
    patches_192.append(p)
    geom_features.append(extract_geometry_feature_vector(
        (p > 0).astype(np.float32), trace_len=GEOMETRY_TRACE_LEN))
    true_labels.append(r['label'])

X_geom = np.array(geom_features, dtype=np.float32)
y_true = np.array([LABEL_MAP[l] for l in true_labels])
print(f'Eval: {len(eval_rows)} junctions | {X_geom.shape[1]} features | {dict(Counter(true_labels))}')

# Eval helper
results = {}
def eval_model(name, y_pred):
    pred_labels = [IDX_TO_LABEL[i] for i in y_pred]
    true_str = [IDX_TO_LABEL[i] for i in y_true]
    print(f'\n{"="*60}\n{name}\n{"="*60}')
    print(classification_report(true_str, pred_labels, labels=['T','Y','X','N'], digits=3, zero_division=0))
    cm = confusion_matrix(true_str, pred_labels, labels=['T','Y','X','N'])
    print(f'Confusion matrix (T/Y/X/N):\n{cm}')
    report = classification_report(true_str, pred_labels, labels=['T','Y','X','N'], output_dict=True, zero_division=0)
    results[name] = {
        'T_f1': report['T']['f1-score'], 'Y_f1': report['Y']['f1-score'],
        'X_f1': report['X']['f1-score'], 'N_f1': report['N']['f1-score'],
        'macro_f1': report['macro avg']['f1-score'], 'accuracy': report.get('accuracy', 0),
    }

---
## 6. Build Stacking Ensemble

The stacking meta-classifier combines three base models:
- **RF** class probabilities (4 dims) — geometry-based
- **XGB** class probabilities (4 dims) — geometry-based
- **Gaussian-masked CNN** class probabilities (4 dims) — pixel-based
- Plus 31-dim geometry features as direct meta-input

Total meta-input: 43 dims → XGB meta-classifier (100 trees, depth 3).

Meta-training uses the internal test split (420 samples) where base model predictions are unbiased (base models never saw this data during training).

In [ ]:
# --- Load base models ---
obj = joblib.load(MODELS / 'RF.joblib')
rf_model = obj['pipeline'] if isinstance(obj, dict) and 'pipeline' in obj else obj

obj = joblib.load(MODELS / 'XGB.joblib')
xgb_model = obj['pipeline'] if isinstance(obj, dict) and 'pipeline' in obj else obj

cnn_model = DeeperCNN_GAP_v2(num_classes=4, in_channels=2, dropout=0.3)
load_init_state(cnn_model, MODELS / 'CNN_finetuned.pt', reset_head=False)
cnn_model.to(device).eval()

# --- Gaussian mask (must match PatchDataset.__getitem__ in train_cnn.py) ---
h = w = CNN_PATCH_SIZE
y_grid, x_grid = np.mgrid[0:h, 0:w]
gauss_mask = np.exp(-((x_grid - h//2)**2 + (y_grid - w//2)**2) / (2 * GAUSSIAN_SIGMA**2)).astype(np.float32)

def get_gauss_cnn_proba(patches):
    """CNN inference with Gaussian center mask.

    Order of operations matches training (PatchDataset.__getitem__):
      1. Normalize skeleton to [0, 1]
      2. Build mask channel via dilation of raw skeleton (threshold 0.5)
      3. Apply Gaussian mask to BOTH channels
    """
    from scipy.ndimage import binary_dilation
    proba = []
    with torch.no_grad():
        for p in patches:
            skel = p.astype(np.float32) / 255.0
            mask = binary_dilation(skel > 0.5, structure=np.ones((3, 3), dtype=bool)).astype(np.float32)
            skel = skel * gauss_mask
            mask = mask * gauss_mask
            inp = torch.tensor(np.stack([skel, mask]),
                               dtype=torch.float32).unsqueeze(0).to(device)
            proba.append(torch.softmax(cnn_model(inp), dim=1).cpu().numpy()[0])
    return np.array(proba)

# --- Load internal test split for meta-training ---
test_rows = [r for r in csv.DictReader(open(NEW_DATA / 'manifest.csv')) if r['split'] == 'test']
test_patches, test_geom, test_y = [], [], []
for r in test_rows:
    p = cv2.imread(str(NEW_DATA / r['relpath']), cv2.IMREAD_GRAYSCALE)
    if p is None: continue
    test_patches.append(p)
    test_geom.append(extract_geometry_feature_vector(
        (p > 0).astype(np.float32), trace_len=GEOMETRY_TRACE_LEN))
    test_y.append(LABEL_MAP[r['label']])
X_test_geom = np.array(test_geom, dtype=np.float32)
y_test = np.array(test_y)
print(f'Meta-training: {len(test_y)} samples')

# --- Base model probabilities ---
rf_p_test = rf_model.predict_proba(X_test_geom)
xgb_p_test = xgb_model.predict_proba(X_test_geom)
cnn_p_test = get_gauss_cnn_proba(test_patches)

rf_p_eval = rf_model.predict_proba(X_geom)
xgb_p_eval = xgb_model.predict_proba(X_geom)
cnn_p_eval = get_gauss_cnn_proba(patches_192)
print('Base model probabilities computed')

# --- Train stacking meta-classifier ---
X_meta_train = np.hstack([rf_p_test, xgb_p_test, cnn_p_test, X_test_geom])
X_meta_eval = np.hstack([rf_p_eval, xgb_p_eval, cnn_p_eval, X_geom])
print(f'Meta-features: {X_meta_train.shape[1]} dims')

meta_clf = XGBClassifier(
    n_estimators=META_ESTIMATORS, max_depth=META_MAX_DEPTH,
    learning_rate=META_LR, eval_metric='mlogloss', random_state=42
)
meta_clf.fit(X_meta_train, y_test)

# --- Evaluate ---
eval_model('Stacking Ensemble', meta_clf.predict(X_meta_eval))

---
## 7. Individual Model Evaluation

Evaluate each base model independently for comparison. This shows the contribution of each component to the ensemble.

In [ ]:
# RF standalone
eval_model('RF (geom-only)', rf_model.predict(X_geom))

# XGB standalone
eval_model('XGB (geom-only)', xgb_model.predict(X_geom))

# CNN standalone (Gaussian-masked)
cnn_preds = np.argmax(cnn_p_eval, axis=1)
eval_model('CNN (Gaussian σ=40)', cnn_preds)

# Majority vote ensemble (for comparison with learned stacking)
vote_preds = []
rf_hard = rf_model.predict(X_geom)
xgb_hard = xgb_model.predict(X_geom)
for i in range(len(y_true)):
    votes = [int(rf_hard[i]), int(xgb_hard[i]), int(cnn_preds[i])]
    vote_preds.append(max(set(votes), key=votes.count))
eval_model('Majority Vote (RF+XGB+CNN)', np.array(vote_preds))

---
## 8. Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T.sort_values('macro_f1', ascending=False)
print('\n' + '='*70)
print('RESULTS: martian Eval (667 junctions)')
print('='*70)
print(df.to_string(float_format='%.3f'))
print(f'\nBest: {df.index[0]}')
print(f'  T F1={df.iloc[0]["T_f1"]:.3f}  Y F1={df.iloc[0]["Y_f1"]:.3f}  '
      f'X F1={df.iloc[0]["X_f1"]:.3f}  N F1={df.iloc[0]["N_f1"]:.3f}')
print(f'  Macro F1={df.iloc[0]["macro_f1"]:.3f}  Accuracy={df.iloc[0]["accuracy"]:.3f}')

---
## 9. Save to Drive

Saves the complete stacking ensemble as a single pickle file, plus individual model checkpoints and results.

In [ ]:
import json

DRIVE_OUT = DRIVE / OUTPUT_DIR_NAME
DRIVE_OUT.mkdir(exist_ok=True)

# Save complete stacking bundle
stacking_bundle = {
    'meta_classifier': meta_clf,
    'rf_model': rf_model,
    'xgb_model': xgb_model,
    'cnn_state_dict': cnn_model.state_dict(),
    'cnn_config': {'arch': CNN_ARCH, 'num_classes': 4, 'in_channels': 2, 'dropout': 0.3},
    'gaussian_sigma': GAUSSIAN_SIGMA,
    'feature_dim': X_geom.shape[1],
    'patch_size': CNN_PATCH_SIZE,
    'label_map': LABEL_MAP,
    'idx_to_label': IDX_TO_LABEL,
    'geometry_trace_len': GEOMETRY_TRACE_LEN,
}
with open(DRIVE_OUT / 'stacking_ensemble.pkl', 'wb') as f:
    pickle.dump(stacking_bundle, f)
print(f'Saved: stacking_ensemble.pkl')

# Save individual checkpoints
for f in MODELS.glob('*'):
    shutil.copy2(f, DRIVE_OUT / f.name)
    print(f'Saved: {f.name} ({f.stat().st_size/1e6:.1f}MB)')

# Save results
with open(DRIVE_OUT / 'eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nAll files saved to: {DRIVE_OUT}')